# 10 Lab — Risk Management as Code

The professional wrapper, made concrete. You will build:

1. A **position-sizing calculator** that turns `analyzer.max_loss` into a share/spread count for a
   fixed-% risk rule.
2. A **portfolio greeks aggregator** across three open DEMO positions, checked against limits.
3. A **checklist-as-code** pre-trade function that refuses a trade failing any rule.

Runs offline, top-to-bottom on the DEMO chain (spot 100, IV ~25%).

In [ ]:
import math
from optionslab import strategies, analyzer, greeks
r = lambda x: round(float(x), 2)
SPOT, VOL = 100.0, 0.26

## 1. Position sizing by max loss

Risk a fixed small % of equity on each trade, sized to the position's **max loss** (not premium):
`units = floor(equity * risk% / |max_loss|)`. Round **down**.

In [ ]:
def size_position(pos, equity, risk_pct):
    ml = analyzer.max_loss(pos)
    if ml == float("-inf") or ml == 0:
        return None, ml                       # undefined risk: size by stress test, not this formula
    dollars_at_risk = equity * risk_pct
    units = math.floor(dollars_at_risk / abs(ml))
    return units, ml

In [ ]:
ic = strategies.iron_condor((87.5,0.37),(92.5,1.01),(107.5,1.19),(112.5,0.43), expiry=45/365)
for eq in (25_000, 50_000):
    units, ml = size_position(ic, eq, 0.02)
    print(f"equity ${eq}: 2% = ${eq*0.02:.0f} at risk, condor max_loss ${r(ml)} -> {units} condor(s)")

On \$25k at 2% you trade **one** condor (`$500 / $360 = 1.38`, rounded down). Note what the
function does with **undefined risk**: a naked short put returns `-inf`, so the formula refuses to
size it — undefined risk must be sized by a stress scenario or avoided.

In [ ]:
naked = strategies.short_put((95, 1.58), expiry=45/365)
print("naked short put units, max_loss:", size_position(naked, 25_000, 0.02))

## 2. Portfolio greeks aggregation

Three open positions. Sum their dollar greeks (`greeks.position_greeks` is addable) to see the
*book's* real exposure — which is often not what any single trade suggests.

In [ ]:
bps = strategies.bull_put_spread((95, 1.58), (90, 0.62), expiry=45/365)
bcs = strategies.bull_call_spread((100, 3.91), (110, 0.73), expiry=45/365)
book = [ic, bps, bcs]
total = greeks.Greeks(0, 0, 0, 0, 0)
for p in book:
    g = greeks.position_greeks(p, SPOT, VOL)
    print(f"{p.label[:30]:30} d={r(g.delta):>7} t={r(g.theta):>6} v={r(g.vega):>6}")
    total = total + g
print(f"{'PORTFOLIO':30} d={r(total.delta):>7} t={r(total.theta):>6} v={r(total.vega):>6}")

This book is **net long delta (+51)** and **net short vega (-8)**: despite one "neutral"
condor, the two bullish spreads dominate direction, and the short-vol condor dominates vega. Check
against limits set while calm.

In [ ]:
LIMITS = {"delta": (-200, 200), "vega": (-50, 50)}
def check_limits(total, limits):
    for k, (lo, hi) in limits.items():
        val = getattr(total, k)
        print(f"net {k} {r(val):>8}  limit [{lo}, {hi}]  ->", "OK" if lo <= val <= hi else "BREACH")
check_limits(total, LIMITS)

Both inside limits here. If net delta drifted past +200, you would hedge it down (short
shares / long puts, module 09) or trim a bullish position before adding more risk.

## 3. Checklist as code

Encode the entry checklist so a trade that fails any rule is refused *before* it is sent. This is the
discipline of module 10 made mechanical.

In [ ]:
def pre_trade_check(pos, spot, vol, equity, risk_pct, iv_rank,
                    is_seller, event_in_horizon, book_total, limits):
    ml = analyzer.max_loss(pos)
    units, _ = size_position(pos, equity, risk_pct)
    proj = book_total + greeks.position_greeks(pos, spot, vol)
    checks = {
        "defined risk & sizable": ml != float("-inf") and units and units >= 1,
        "IV column correct": (iv_rank >= 50) == is_seller,
        "no event in horizon": not event_in_horizon,
        "delta limit after": limits["delta"][0] <= proj.delta <= limits["delta"][1],
        "vega limit after": limits["vega"][0] <= proj.vega <= limits["vega"][1],
    }
    return checks, all(checks.values())

In [ ]:
# A high-IV seller's condor with no event, correctly sized -> should pass
checks, ok = pre_trade_check(ic, SPOT, VOL, 25_000, 0.02, iv_rank=65,
                             is_seller=True, event_in_horizon=False,
                             book_total=greeks.Greeks(0,0,0,0,0), limits=LIMITS)
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print("SEND TRADE" if ok else "DO NOT SEND")

Now break a rule: the same condor (a premium *seller*) when **IV rank is low (12)** — you
would be selling cheap options, the wrong side of the vol column. The checklist should refuse it.

In [ ]:
checks, ok = pre_trade_check(ic, SPOT, VOL, 25_000, 0.02, iv_rank=12,
                             is_seller=True, event_in_horizon=False,
                             book_total=greeks.Greeks(0,0,0,0,0), limits=LIMITS)
print({k: ("PASS" if v else "FAIL") for k, v in checks.items()})
print("SEND TRADE" if ok else "DO NOT SEND")

## Experiments

1. Change the sizing rule to 1% and 5% risk. How does the condor count change on a \$50k account?
2. Add a fourth position (e.g., a long calendar or long call) and re-aggregate. Can you push net vega
   back toward zero with a long-vega position as ballast?
3. Tighten `LIMITS["delta"]` to (-100, 100). Does the current book still pass? What would you trim?
4. Add an `event_in_horizon=True` flag to the passing condor — confirm the checklist now refuses it.
5. Extend `pre_trade_check` with a "liquidity passed" boolean argument and require it. Real trades
   fail here most often.